# Object Detection with KerasCV — Student Worksheet

**Total: 100 points**

You will implement:
- Loading and visualising images from a provided COCO subset
- Running a pre-trained YOLOv8 Pascal VOC detector and controlling NMS
- Implementing Intersection over Union (IoU) from scratch
- Parsing ground-truth annotations and building a tf.data evaluation pipeline
- Computing COCO evaluation metrics with `BoxCOCOMetrics`
- Computing AP@0.50 for the `person` class step by step

## Important rules

Use the **exact variable names** requested in each part — the grader depends on them.

Keep all **sanity-check `print(...)` lines** in place; do not remove them.

When a cell starts with:
```python
variable = None  # <-- replace
```
replace `None` with your solution.

Do **not** use `raise NotImplementedError`.

In [ ]:
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import keras_cv
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow :', tf.__version__)
print('KerasCV    :', keras_cv.__version__)

In [ ]:
# ── Fill in your path ────────────────────────────────────────────────────────
IMG_DIR    = 'coco/'               # folder containing the 10 COCO .jpg images
ANNOT_FILE = 'coco/annotations.json'   # provided alongside the worksheet
# ─────────────────────────────────────────────────────────────────────────────

IMG_SIZE = (640, 640)
BBOX_FMT = 'xyxy'
N_IMGS   = 10

# load annotation file once — keys are the image filenames
ann    = json.load(open(ANNOT_FILE))
fnames = sorted(ann.keys())
print(f'{len(fnames)} images:', fnames)

In [ ]:
CLASS_MAP = {
     0: 'aeroplane',  1: 'bicycle',    2: 'bird',       3: 'boat',
     4: 'bottle',     5: 'bus',        6: 'car',        7: 'cat',
     8: 'chair',      9: 'cow',       10: 'diningtable',11: 'dog',
    12: 'horse',     13: 'motorbike', 14: 'person',     15: 'pottedplant',
    16: 'sheep',     17: 'sofa',      18: 'train',      19: 'tvmonitor',
}
CMAP_INV = {v: k for k, v in CLASS_MAP.items()}   # name → id


def set_nms(mdl, iou_thresh=0.5, conf_thresh=0.2, max_det=50):
    """Replace the NMS decoder of mdl with the given thresholds."""
    mdl.prediction_decoder = keras_cv.layers.NonMaxSuppression(
        bounding_box_format=BBOX_FMT,
        from_logits=False,
        max_detections=max_det,
        iou_threshold=iou_thresh,
        confidence_threshold=conf_thresh,
    )
    print(f'NMS: iou_threshold={iou_thresh}  confidence_threshold={conf_thresh}')

## Part 1 — Pre-trained YOLOv8 Inference (35 pts)

### 1.1 Load Images  *(10 pts)*

The filenames are already in `fnames` (sorted keys of `ann`).
Load each image from `IMG_DIR`, decode it, cast to `float32`,
resize with padding to `(640, 640)`, and stack into a single tensor.

Required variables:
- `images` — `tf.Tensor` of shape `(10, 640, 640, 3)`, dtype `float32`, range `[0, 255]`

In [ ]:
# TODO (1.1): load images
# resize_with_pad mantém o aspect ratio enchendo com preto
images = tf.stack([
    tf.image.resize_with_pad(
        tf.cast(tf.image.decode_jpeg(tf.io.read_file(IMG_DIR + f), channels=3), tf.float32),
        IMG_SIZE[0], IMG_SIZE[1]
    )
    for f in fnames
])

# Sanity checks (do not remove)
print('images shape:', images.shape)
print('images dtype:', images.dtype)

### 1.2 Visualise Images

Display all 10 images in a 5 × 2 grid. Label each with its filename.

In [ ]:
# TODO (1.2): visualise all 10 images in a 5×2 grid
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for ax, img, fname in zip(axes.flat, images.numpy() / 255., fnames):
    ax.imshow(img)
    ax.set_title(fname, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

### 1.3 Load Model and Run Predictions  *(15 pts)*

Load the **YOLOv8-M Pascal VOC** pre-trained model and run inference on `images`.

Required variables:
- `model`      — `YOLOV8Detector` from preset `'yolo_v8_m_pascalvoc'`
- `detections` — dict with keys `'boxes'`, `'confidence'`, `'classes'`

In [ ]:
# TODO (1.3): load model and predict
model = keras_cv.models.YOLOV8Detector.from_preset(
    'yolo_v8_m_pascalvoc',
    bounding_box_format=BBOX_FMT,
)
detections = model.predict(images)

# Sanity checks (do not remove)
print('model type       :', type(model).__name__)
print('boxes shape      :', detections['boxes'].shape)
print('confidence shape :', detections['confidence'].shape)

### 1.4 Visualise Detections

Draw predicted bounding boxes on all 10 images (5 × 2 grid).
Show only detections with confidence > 0.2. Label each box with class name and score.

In [ ]:
# TODO (1.4): visualise detections (confidence > 0.2)
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for ax, i in zip(axes.flat, range(N_IMGS)):
    ax.imshow(images[i].numpy() / 255.)
    boxes  = detections['boxes'][i].numpy()
    confs  = detections['confidence'][i].numpy()
    clsids = detections['classes'][i].numpy().astype(int)
    for box, conf, cls_id in zip(boxes, confs, clsids):
        if conf < 0.2:
            continue
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                              linewidth=1.5, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-3, f'{CLASS_MAP.get(cls_id, cls_id)} {conf:.2f}',
                color='red', fontsize=6, va='bottom')
    ax.axis('off')
plt.tight_layout()
plt.show()

### 1.5 Fewer Bounding Boxes  *(5 pts)*

Use `set_nms` to reconfigure the model so that **fewer** bounding boxes survive,
then re-run `model.predict`.

Hint: raising `conf_thresh` discards low-confidence boxes;
lowering `iou_thresh` makes NMS more aggressive.

Required variable: `det_few`

In [ ]:
# TODO (1.5): stricter NMS → det_few
# conf_thresh alto descarta caixas de baixa confiança; iou_thresh baixo é NMS mais agressivo
set_nms(model, iou_thresh=0.3, conf_thresh=0.5)
det_few = model.predict(images)

print('det_few boxes shape:', det_few['boxes'].shape)

### 1.6 More Bounding Boxes  *(5 pts)*

Use `set_nms` to let **more** bounding boxes survive, then re-run.

Required variable: `det_many`

In [ ]:
# TODO (1.6): permissive NMS → det_many
# conf_thresh baixo aceita caixas fracas; iou_thresh alto permite mais sobreposição
set_nms(model, iou_thresh=0.7, conf_thresh=0.05)
det_many = model.predict(images)

print('det_many boxes shape:', det_many['boxes'].shape)

### 1.7 Visualise the Comparison  *(optional)*

Show `det_few` and `det_many` side by side for one image of your choice.

In [ ]:
# (optional) compare det_few vs det_many for image 0
IMG_IDX = 0
img = images[IMG_IDX].numpy() / 255.

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
for ax, dets, title in [(ax1, det_few, 'Fewer boxes (strict NMS)'),
                         (ax2, det_many, 'More boxes (permissive NMS)')]:
    ax.imshow(img)
    for box, conf, cls_id in zip(dets['boxes'][IMG_IDX].numpy(),
                                  dets['confidence'][IMG_IDX].numpy(),
                                  dets['classes'][IMG_IDX].numpy().astype(int)):
        if conf < 0.05:
            continue
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1.5,
                              edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-3, f'{CLASS_MAP.get(cls_id,"?")} {conf:.2f}',
                color='lime', fontsize=7, va='bottom')
    ax.set_title(title); ax.axis('off')
plt.tight_layout()
plt.show()

## Part 2 — Intersection over Union  *(10 pts)*

IoU (Intersection over Union) is the ratio of the overlapping area to the
combined area of two bounding boxes.  It is used in two places in this worksheet:

* **NMS** suppresses a box when its IoU with a higher-confidence box exceeds
  `iou_threshold`.
* **TP / FP assignment** (Part 4.4): a prediction counts as a True Positive only
  when its IoU with an unmatched GT box is ≥ `IOU_THR = 0.50`.

The helper `box_iou_matrix` (given below) computes pairwise IoU between two
sets of boxes.  Apply it to the two example predictions below and decide
whether each one would be a **TP** or **FP** at the standard threshold.

In [ ]:
# Given — do not modify ──────────────────────────────────────────────────────
def box_iou_matrix(a, b):
    """Pairwise IoU between arrays a (M,4) and b (N,4) in xyxy format."""
    x1    = np.maximum(a[:, 0:1], b[:, 0]);  y1 = np.maximum(a[:, 1:2], b[:, 1])
    x2    = np.minimum(a[:, 2:3], b[:, 2]);  y2 = np.minimum(a[:, 3:4], b[:, 3])
    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


# Example boxes (640×640 letterboxed image space, xyxy)
GT_BOX = np.array([[ 50., 100., 250., 400.]])   # one GT person box
PRED_A = np.array([[ 60., 110., 240., 390.]])   # prediction A — large overlap
PRED_B = np.array([[300., 100., 500., 400.]])   # prediction B — no overlap

iou_A = float(box_iou_matrix(PRED_A, GT_BOX)[0, 0])
iou_B = float(box_iou_matrix(PRED_B, GT_BOX)[0, 0])
print(f'IoU(PRED_A, GT_BOX) = {iou_A:.4f}')
print(f'IoU(PRED_B, GT_BOX) = {iou_B:.4f}')
# ─────────────────────────────────────────────────────────────────────────────

# TODO (2.1): set is_tp_A and is_tp_B (True / False) for IOU_THR = 0.50
IOU_THR = 0.50

# Uma predição é TP se o seu IoU com a melhor GT box >= threshold
is_tp_A = iou_A >= IOU_THR   # True  — grande sobreposição
is_tp_B = iou_B >= IOU_THR   # False — sem sobreposição

# Sanity checks (do not remove)
_fmt = lambda v: 'TP' if v else ('FP' if v is not None else '?')
print(f'PRED_A  iou={iou_A:.4f}  →  {_fmt(is_tp_A)}')
print(f'PRED_B  iou={iou_B:.4f}  →  {_fmt(is_tp_B)}')

## Part 3 — Ground-Truth Annotations  *(20 pts)*

### 3.1 Parse the Annotations  *(10 pts)*

`ann` is already loaded (see config cell). Each entry looks like:
```python
ann['000000031599.jpg'] = {
    'boxes':   [[x1, y1, x2, y2], ...],
    'classes': ['boat', 'boat', ...]
}
```

Required variables:
- `gt_boxes`   — list of 10 inner lists, each `[[x1,y1,x2,y2], …]` as **floats**
- `gt_classes` — list of 10 inner lists, each containing **integer** class ids
  (use `CMAP_INV` to convert name → id; keep the same order as `fnames`)

In [ ]:
# TODO (3.1): parse gt_boxes and gt_classes from ann
# Boxes como floats, classes convertidas de nome para id inteiro via CMAP_INV
gt_boxes   = [[list(map(float, b)) for b in ann[f]['boxes']]   for f in fnames]
gt_classes = [[CMAP_INV[c]         for c in ann[f]['classes']] for f in fnames]

# Sanity checks (do not remove)
print('gt_boxes[0] count:', len(gt_boxes[0]))
print('gt_classes[0]    :', gt_classes[0][:4])

### 3.2 Per-Class GT Count Table  *(10 pts)*

Count total GT boxes per class across all 10 images and display as a sorted table.

Required variable:
- `class_counts` — `dict` mapping class **name** → total GT box count (classes with count > 0)

In [ ]:
# TODO (3.2): build class_counts and display table
# Contar GT boxes por nome de classe em todas as 10 imagens
class_counts = defaultdict(int)
for classes in gt_classes:
    for cls_id in classes:
        class_counts[CLASS_MAP[cls_id]] += 1
class_counts = dict(class_counts)

if class_counts is not None:
    _df = pd.DataFrame(sorted(class_counts.items(), key=lambda x: -x[1]),
                       columns=['Class', 'GT boxes'])
    display(_df)
print('class_counts type:', type(class_counts))

## Part 4 — Evaluation Pipeline  *(35 pts)*

### 4.1 Build the tf.data Evaluation Pipeline  *(10 pts)*

Use the helpers below to build a pipeline that reads each image + GT from disk,
applies the KerasCV `Resizing` layer (which also transforms the bounding boxes),
and collects everything into a single batch of 10.

Required variables:
- `I_ev` — `float32` tensor of shape `(10, 640, 640, 3)`
- `b_ev` — dict with keys `'boxes'` and `'classes'` (both `RaggedTensor`)

In [ ]:
# Given helpers — do not modify
imResize = keras_cv.layers.Resizing(
    *IMG_SIZE, pad_to_aspect_ratio=True, bounding_box_format=BBOX_FMT)

def _load_image_bb(path, boxes, classes):
    raw = tf.io.read_file(path)
    img = tf.cast(tf.image.decode_jpeg(raw, channels=3), tf.float32)
    return img, {'boxes': boxes, 'classes': classes}

def _resize_image_bb(img, bb):
    d = imResize({'images': img, 'bounding_boxes': bb})
    return d['images'], d['bounding_boxes']

In [ ]:
# TODO (4.1): build the tf.data pipeline → I_ev, b_ev
paths     = [IMG_DIR + f for f in fnames]
# RaggedTensors porque cada imagem tem número diferente de boxes
boxes_r   = tf.ragged.constant(gt_boxes, dtype=tf.float32)
classes_r = tf.ragged.constant([[float(c) for c in row] for row in gt_classes],
                                dtype=tf.float32)

ds_eval = (
    tf.data.Dataset.from_tensor_slices((paths, boxes_r, classes_r))
    .map(_load_image_bb)
    .ragged_batch(N_IMGS)
    .map(_resize_image_bb)
    .cache()
)

I_ev, b_ev = next(iter(ds_eval))

print('I_ev shape      :', I_ev.shape)
print('b_ev boxes type :', type(b_ev['boxes']).__name__)

### 4.2 Visualise GT vs Predicted

Display all 10 images with **green** GT boxes and **orange** predictions (confidence > 0.2).

In [ ]:
# TODO (4.2): GT (green) vs predictions (orange) for all 10 images
set_nms(model, iou_thresh=0.5, conf_thresh=0.2)
preds_viz = model.predict(I_ev, verbose=0)

fig, axes = plt.subplots(2, 5, figsize=(18, 9))
for ax, i in zip(axes.flat, range(N_IMGS)):
    ax.imshow(I_ev[i].numpy() / 255.)

    # Ground truth — verde
    for box, cls_id in zip(b_ev['boxes'][i].numpy(),
                            b_ev['classes'][i].numpy().astype(int)):
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1.5,
                              edgecolor='green', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-3, CLASS_MAP.get(cls_id, '?'),
                color='green', fontsize=6, va='bottom')

    # Predições — laranja
    for box, conf, cls_id in zip(preds_viz['boxes'][i].numpy(),
                                  preds_viz['confidence'][i].numpy(),
                                  preds_viz['classes'][i].numpy().astype(int)):
        if conf < 0.2:
            continue
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1.5,
                              edgecolor='orange', facecolor='none')
        ax.add_patch(rect)
        ax.text(x2, y2+9, f'{CLASS_MAP.get(cls_id,"?")} {conf:.2f}',
                color='orange', fontsize=6, va='top', ha='right')
    ax.axis('off')
plt.suptitle('Verde = GT   Laranja = predições (conf > 0.2)', fontsize=11)
plt.tight_layout()
plt.show()

### 4.3 COCO Evaluation Metrics  *(15 pts)*

Use `keras_cv.metrics.BoxCOCOMetrics` to compute mAP on the 10-image set.
Because images have different numbers of GT boxes, pad them to a fixed dense
shape with the helper `pad_gt_to_fixed` provided below.

Required variables:
- `model_eval`   — fresh `YOLOV8Detector` (same preset)
- `coco_metric`  — `BoxCOCOMetrics` instance after calling `update_state`
- `eval_results` — dict from `coco_metric.result()`
- `mAP50`        — `float`, value at key `'MaP@.50IOU'`

In [ ]:
# Given — do not modify
def pad_gt_to_fixed(boxes_ragged, classes_ragged, max_boxes=100):
    """Pad ragged GT tensors to fixed dense shape required by BoxCOCOMetrics."""
    boxes_d   = boxes_ragged.to_tensor(default_value=-1.,
                                        shape=[None, max_boxes, 4])
    classes_d = classes_ragged.to_tensor(default_value=-1.,
                                          shape=[None, max_boxes])
    return boxes_d, classes_d

In [ ]:
# TODO (4.3): compute COCO metrics
# Modelo fresco com NMS default para avaliação justa
model_eval = keras_cv.models.YOLOV8Detector.from_preset(
    'yolo_v8_m_pascalvoc',
    bounding_box_format=BBOX_FMT,
)
preds_eval = model_eval.predict(I_ev, verbose=0)

# BoxCOCOMetrics requer tensores densos — padding das GT ragged com -1
boxes_d, classes_d = pad_gt_to_fixed(b_ev['boxes'], b_ev['classes'])
y_true = {'boxes': boxes_d, 'classes': classes_d}

coco_metric = keras_cv.metrics.BoxCOCOMetrics(
    bounding_box_format=BBOX_FMT,
    evaluate_freq=1,
)
coco_metric.update_state(y_true, preds_eval)

eval_results = coco_metric.result()
mAP50        = float(eval_results['MaP@.50IOU'])

print('eval_results keys:', list(eval_results.keys()))
print('mAP@0.50         :', mAP50)

### 4.4 AP@0.50 for `person`  *(10 pts)*

Compute Average Precision for the **`person`** class (id = 14) following the
Pascal VOC 2010+ protocol:

1. Re-run with `EVAL_THR = 0.05` to capture the full P/R curve.
2. Collect all `person` predictions across 10 images; sort by confidence descending.
3. Greedy IoU matching (`IOU_THR = 0.50`): label each detection TP or FP.
   A GT box can only be matched once.
4. Compute cumulative precision and recall at each step.
5. VOC monotone interpolation: smooth precision right-to-left with a running max,
   then `AP = Σ (recall[i+1] − recall[i]) × precision[i+1]` for unique recall changes.
6. Plot the P/R curve.

In [ ]:
# Given — do not modify
def box_iou_matrix(a, b):
    """Pairwise IoU between arrays a (M,4) and b (N,4) in xyxy format."""
    x1    = np.maximum(a[:, 0:1], b[:, 0]);   y1 = np.maximum(a[:, 1:2], b[:, 1])
    x2    = np.minimum(a[:, 2:3], b[:, 2]);   y2 = np.minimum(a[:, 3:4], b[:, 3])
    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

In [ ]:
# TODO (4.4a): collect person detections and assign TP / FP
IOU_THR   = 0.50
EVAL_THR  = 0.05
PERSON_ID = CMAP_INV['person']

# Threshold baixo para capturar a curva P/R completa
set_nms(model, iou_thresh=0.5, conf_thresh=EVAL_THR)
preds_all = model.predict(I_ev, verbose=0)

n_gt_person = sum(gt_classes[i].count(PERSON_ID) for i in range(N_IMGS))

person_dets = []
for img_i in range(N_IMGS):
    # GT boxes de pessoa nesta imagem
    gt_pb = np.array([gt_boxes[img_i][j]
                      for j, c in enumerate(gt_classes[img_i]) if c == PERSON_ID],
                     dtype=np.float32).reshape(-1, 4)
    matched = set()  # índices de GT já emparelhados (cada GT só conta uma vez)

    boxes_i = preds_all['boxes'][img_i].numpy()
    conf_i  = preds_all['confidence'][img_i].numpy()
    cls_i   = preds_all['classes'][img_i].numpy().astype(int)

    for box, conf, cls_id in zip(boxes_i, conf_i, cls_i):
        if cls_id != PERSON_ID or conf < EVAL_THR:
            continue
        is_tp = False
        if len(gt_pb) > 0:
            ious = box_iou_matrix(box[None], gt_pb)[0]
            best = int(np.argmax(ious))
            if ious[best] >= IOU_THR and best not in matched:
                is_tp = True
                matched.add(best)
        person_dets.append({'img_i': img_i, 'conf': conf, 'box': box, 'is_tp': is_tp})

if person_dets is not None:
    n_tp = sum(1 for d in person_dets if d.get('is_tp'))
    print(f'{len(person_dets)} person detections: {n_tp} TP, {len(person_dets)-n_tp} FP')
print(f'GT person boxes: {n_gt_person}')

In [ ]:
# TODO (4.4b): AP computation and P/R plot
# Ordenar por confiança decrescente para construir a curva P/R
person_dets.sort(key=lambda d: -d['conf'])

tp_arr = np.array([d['is_tp'] for d in person_dets], dtype=float)
fp_arr = 1.0 - tp_arr

cum_tp = np.cumsum(tp_arr)
cum_fp = np.cumsum(fp_arr)

precision = cum_tp / (cum_tp + cum_fp + 1e-9)
recall    = cum_tp / (n_gt_person + 1e-9)

# Interpolação monotónica VOC 2010+: running max da direita para a esquerda
precision_smooth = np.maximum.accumulate(precision[::-1])[::-1]

# AP = área sob a curva P/R interpolada (regra rectangular)
ap_person = float(np.sum(np.diff(recall) * precision_smooth[1:]))

plt.figure(figsize=(6, 5))
plt.step(recall, precision_smooth, 'b-', linewidth=2, where='post')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title(f'P/R Curve — person class  (AP@0.50 = {ap_person:.3f})')
plt.xlim(0, 1); plt.ylim(0, 1.05)
plt.grid(True)
plt.tight_layout()
plt.show()

print(f'AP@0.50 (person) = {ap_person:.4f}')

## Auto-grader

Run the cell below to check your score.

In [ ]:
from grader_objectDetect import grade
grade(globals())